# R3maJ Kaggle Notebook v4

Kaggle T4×2 training setup for R3maJ with Google Drive replay/checkpoint restore, **automatic checkpoint backup to Google Drive**, and **256 games**.

The notebook restores the shared `R3maJ` Drive folder at startup. During training, a background watcher detects newly created/updated checkpoints and uploads them to the Drive `checkpoints/` folder.


In [ ]:
# 1. Clone R3maJ
import os, subprocess
ROOT='/kaggle/working/R3maJ'
REPO='https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT,'.git')):
    subprocess.run(['git','clone','--depth','1',REPO,ROOT],check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'],check=False)
print(ROOT)


In [ ]:
# 2. Download the complete R3maJ Drive folder (replay + checkpoints)
!pip install -q gdown

import gdown, os, shutil
DRIVE_FOLDER_ID='1ktqAHT6wwYCyyRA4REBN2kZyFqeCbvKh'
DOWNLOAD_DIR='/kaggle/working/R3maJ_drive'
LOCAL_ROOT='/kaggle/working/R3maJ/build'
LOCAL_REPLAY=f'{LOCAL_ROOT}/serialized_replays.bin'
LOCAL_CHECKPOINTS=f'{LOCAL_ROOT}/checkpoints'
os.makedirs(LOCAL_ROOT,exist_ok=True)
if not os.path.exists(DOWNLOAD_DIR):
    gdown.download_folder(f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}',output=DOWNLOAD_DIR,quiet=False,use_cookies=False)
print('Drive folder downloaded to:',DOWNLOAD_DIR)
for root,dirs,files in os.walk(DOWNLOAD_DIR):
    level=root.replace(DOWNLOAD_DIR,'').count(os.sep)
    indent='  '*level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files[:20]: print(f'{indent}  {f}')


In [ ]:
# 3. Restore replay + checkpoints into the locations R3maJ expects
import os, shutil
def find_file(base,name):
    for root,dirs,files in os.walk(base):
        if name in files: return os.path.join(root,name)
    return None
def find_dir(base,name):
    for root,dirs,files in os.walk(base):
        if name in dirs: return os.path.join(root,name)
    return None
src_replay=find_file(DOWNLOAD_DIR,'serialized_replays.bin')
src_checkpoints=find_dir(DOWNLOAD_DIR,'checkpoints')
assert src_replay,'serialized_replays.bin not found in shared R3maJ folder.'
assert src_checkpoints,'checkpoints folder not found in shared R3maJ folder.'
shutil.copy2(src_replay,LOCAL_REPLAY)
if os.path.exists(LOCAL_CHECKPOINTS): shutil.rmtree(LOCAL_CHECKPOINTS)
shutil.copytree(src_checkpoints,LOCAL_CHECKPOINTS)
print('Replay:',LOCAL_REPLAY)
print('Replay GB:',round(os.path.getsize(LOCAL_REPLAY)/(1024**3),3))
print('Checkpoint entries:',sorted(os.listdir(LOCAL_CHECKPOINTS))[:20])


### Drive layout

`R3maJ/serialized_replays.bin`
`R3maJ/checkpoints/<checkpoint directories>`

The single folder ID above restores both.


In [ ]:
# 4. Install build dependencies + inspect both GPUs
import subprocess,os
subprocess.run(['apt-get','update','-qq'],check=False)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','git','libpython3-dev','pkg-config'],check=False)
import torch
print('torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),torch.version.cuda)
print('GPU count:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
assert torch.cuda.device_count() >= 2,'Expected Kaggle T4×2 runtime.'
subprocess.run(['nvidia-smi'],check=False)


In [ ]:
# 5. Configure + build
import os,subprocess,torch
os.chdir(ROOT)
prefix=os.path.dirname(torch.__file__)
subprocess.run(['cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release',f'-DTORCH_INSTALL_PREFIX={prefix}'],check=True)
subprocess.run(['cmake','--build','build','-j',str(os.cpu_count() or 2)],check=True)
EXE=os.path.join(ROOT,'build','R3maJ')
print('Binary:',EXE,os.path.exists(EXE))


In [ ]:
# 6. Verify restored data
assert os.path.exists(EXE),'R3maJ binary missing.'
assert os.path.exists(LOCAL_REPLAY),'Replay missing.'
assert os.path.isdir(LOCAL_CHECKPOINTS),'Checkpoint directory missing.'
print('READY')
print('Games:',256)
print('Checkpoint entries:',len([x for x in os.listdir(LOCAL_CHECKPOINTS) if not x.startswith('.')]))


## Google Drive checkpoint backup

Kaggle cannot upload to Google Drive with `gdown`. This notebook therefore uses the Google Drive API for **Kaggle → Drive** checkpoint backup.

### One-time setup

Create a Kaggle Secret named **`GOOGLE_DRIVE_TOKEN_JSON`** containing an OAuth authorized-user JSON for a Google account that has write access to the shared `R3maJ` folder.

The secret value must be the full JSON containing at least `client_id`, `client_secret`, `refresh_token`, and `token`.

The watcher checks the local checkpoint tree every **60 seconds**. When R3maJ finishes writing a new checkpoint, it uploads/synchronizes it to Drive. Therefore if R3maJ saves every **5M steps**, those checkpoints will be backed up shortly after each 5M-step save.

**Do not put OAuth credentials directly in this notebook or GitHub.**


In [ ]:
# 7. Install Google Drive API client
!pip install -q google-api-python-client google-auth google-auth-httplib2


In [ ]:
# 8. Authenticate and prepare Drive checkpoint sync
import os, json, io, time, hashlib, threading, shutil
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth.transport.requests import Request
from googleapiclient.errors import HttpError

try:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    token_info = json.loads(secret_client.get_secret("GOOGLE_DRIVE_TOKEN_JSON"))
except Exception as e:
    raise RuntimeError(
        "Missing Kaggle Secret GOOGLE_DRIVE_TOKEN_JSON. Add an OAuth authorized-user JSON for a Google account with write access to the R3maJ Drive folder."
    ) from e

SCOPES=['https://www.googleapis.com/auth/drive']
creds=Credentials.from_authorized_user_info(token_info,SCOPES)
if creds.expired and creds.refresh_token:
    creds.refresh(Request())
drive=build('drive','v3',credentials=creds,cache_discovery=False)
DRIVE_ROOT_ID='1ktqAHT6wwYCyyRA4REBN2kZyFqeCbvKh'

def drive_find_child(name,parent_id,mime_type=None):
    safe_name=name.replace("'","\\'")
    q=f"'{parent_id}' in parents and name = '{safe_name}' and trashed = false"
    if mime_type:
        q += f" and mimeType = '{mime_type}'"
    res=drive.files().list(q=q,fields='files(id,name,mimeType,modifiedTime)',pageSize=100).execute()
    return res.get('files',[])

checkpoint_folders=drive_find_child('checkpoints',DRIVE_ROOT_ID,'application/vnd.google-apps.folder')
if checkpoint_folders:
    DRIVE_CHECKPOINTS_ID=checkpoint_folders[0]['id']
else:
    meta={'name':'checkpoints','mimeType':'application/vnd.google-apps.folder','parents':[DRIVE_ROOT_ID]}
    DRIVE_CHECKPOINTS_ID=drive.files().create(body=meta,fields='id').execute()['id']
print('Drive checkpoint folder ID:',DRIVE_CHECKPOINTS_ID)
print('Drive authentication: OK')


In [ ]:
# 9. Background checkpoint watcher: Kaggle -> Google Drive
SYNC_INTERVAL=60
_stop_backup=False
_last_uploaded={}

def drive_find_child(name,parent_id,mime_type=None):
    safe_name=name.replace("'","\\'")
    q=f"'{parent_id}' in parents and name = '{safe_name}' and trashed = false"
    if mime_type:
        q += f" and mimeType = '{mime_type}'"
    res=drive.files().list(q=q,fields='files(id,name,mimeType,modifiedTime)',pageSize=100).execute()
    return res.get('files',[])

def ensure_drive_folder(name,parent_id):
    found=drive_find_child(name,parent_id,'application/vnd.google-apps.folder')
    if found: return found[0]['id']
    meta={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]}
    return drive.files().create(body=meta,fields='id').execute()['id']

def upload_file_to_drive(local_path,parent_id,name):
    found=drive_find_child(name,parent_id)
    media=MediaFileUpload(local_path,resumable=True)
    if found:
        drive.files().update(fileId=found[0]['id'],media_body=media).execute()
    else:
        meta={'name':name,'parents':[parent_id]}
        drive.files().create(body=meta,media_body=media,fields='id').execute()

def sync_checkpoint_tree():
    if not os.path.isdir(LOCAL_CHECKPOINTS): return
    for root,dirs,files in os.walk(LOCAL_CHECKPOINTS):
        rel_root=os.path.relpath(root,LOCAL_CHECKPOINTS)
        parent_id=DRIVE_CHECKPOINTS_ID
        if rel_root != '.':
            for part in rel_root.split(os.sep):
                parent_id=ensure_drive_folder(part,parent_id)
        for name in files:
            local_path=os.path.join(root,name)
            try:
                sig=(os.path.getsize(local_path),os.path.getmtime_ns(local_path))
                key=os.path.relpath(local_path,LOCAL_CHECKPOINTS)
                if _last_uploaded.get(key)==sig: continue
                time.sleep(0.05)
                if sig != (os.path.getsize(local_path),os.path.getmtime_ns(local_path)): continue
                upload_file_to_drive(local_path,parent_id,name)
                _last_uploaded[key]=sig
                print(f'[Drive backup] uploaded: {key}')
            except FileNotFoundError:
                continue

def backup_loop():
    print(f'[Drive backup] watcher started; checking every {SYNC_INTERVAL}s')
    while not _stop_backup:
        try: sync_checkpoint_tree()
        except Exception as e: print('[Drive backup] ERROR:',repr(e))
        for _ in range(SYNC_INTERVAL):
            if _stop_backup: break
            time.sleep(1)

backup_thread=threading.Thread(target=backup_loop,daemon=True)
backup_thread.start()


In [ ]:
# 10. Start training — 256 games
import os,subprocess
os.chdir(os.path.join(ROOT,'build'))
TRAIN_ARGS=['--device','cuda','--save-dir','checkpoints','--games','256','--replays','serialized_replays.bin']
print('Launching:','./R3maJ',*TRAIN_ARGS)
proc=subprocess.Popen(['./R3maJ']+TRAIN_ARGS)
print('Training PID:',proc.pid)


## T4×2 note

Kaggle provides two Tesla T4 GPUs. The current R3maJ executable has no verified multi-GPU collector/learner option, so v4 keeps the safe single-process `--device cuda` configuration.

The important addition here is **checkpoint persistence**: the background watcher continuously mirrors completed checkpoints from Kaggle to Google Drive.

If the Kaggle session dies, the next session downloads the Drive checkpoint tree first and resumes from the persisted checkpoint state.
